In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Baseline Model Kurs (Time-Series Only)

# 1. Load & Sortir Data

In [11]:
bi_raw = pd.read_csv('../data/raw/bi-usd-rate.csv')
print(f"Baris awal                : {len(bi_raw)}")
bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")

bi_raw = bi_raw.sort_values("Tanggal_dt").reset_index(drop=True)

print(f"Rentang tanggal            : {bi_raw['Tanggal_dt'].min().date()} s.d. {bi_raw['Tanggal_dt'].max().date()}")
bi_raw.head(5)

Baris awal                : 1202
Rentang tanggal            : 2021-09-01 s.d. 2026-09-01


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_29048\316416140.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal,Tanggal_dt
0,1202,1,14377.53,14234.47,9/1/2021 12:00:00 AM,2021-09-01
1,1201,1,14355.42,14212.58,9/2/2021 12:00:00 AM,2021-09-02
2,1200,1,14352.41,14209.60,9/3/2021 12:00:00 AM,2021-09-03
3,1199,1,14332.31,14189.70,9/6/2021 12:00:00 AM,2021-09-06
4,1198,1,14310.20,14167.81,9/7/2021 12:00:00 AM,2021-09-07


# 2. Kurs Tengah

In [12]:
bi_raw["Kurs Tengah"] = (bi_raw["Kurs Jual"] + bi_raw["Kurs Beli"]) / 2
bi_raw[["Tanggal_dt", "Kurs Jual", "Kurs Beli", "Kurs Tengah"]].head(5)

bi_raw = bi_raw.drop(columns=["Kurs Jual", "Kurs Beli"])

# 3. Formulasi Target (NAIK/TURUN)

In [13]:
df_model = bi_raw.copy()

#target: apakah kurs tengah hari trading berikutnya lebih tinggi dari hari ini
df_model["kurs Next"] = df_model["Kurs Tengah"].shift(-1)
df_model["Target"] = (df_model["kurs Next"] > df_model["Kurs Tengah"]).astype(int)

#baris terakhir tidak punya kurs_next, dibuang
df_model = df_model.iloc[:-1].copy()

print("Distribusi target (1=NAIK, 0=TURUN/TETAP):")
print(df_model["Target"].value_counts(normalize=True))

df_model.tail(5)

Distribusi target (1=NAIK, 0=TURUN/TETAP):
Target
1    0.553705
0    0.446295
Name: proportion, dtype: float64


,NO,Nilai,Tanggal,Tanggal_dt,Kurs Tengah,kurs Next,Target
1196,6,1,8/24/2026 12:00:00 AM,2026-08-24,17705.0,17703.0,0
1197,5,1,8/26/2026 12:00:00 AM,2026-08-26,17703.0,17717.0,1
1198,4,1,8/27/2026 12:00:00 AM,2026-08-27,17717.0,17703.0,0
1199,2,1,8/31/2026 12:00:00 AM,2026-08-31,17703.0,17746.0,1
1200,1,1,9/1/2026 12:00:00 AM,2026-09-01,17746.0,17762.0,1


# 4. Fitur Time-Series (Lag & Rolling)

In [14]:
for lag in range(1, 6):
    df_model[f"kurs_lag{lag}"] = df_model["Kurs Tengah"].shift(lag)

#rolling mean & std 5 hari, dihitung dari shift(1) supaya tidak memakai kurs hari yang sama dengan target
df_model["kurs_roll_mean5"] = df_model["Kurs Tengah"].shift(1).rolling(5).mean()
df_model["kurs_roll_std5"] = df_model["Kurs Tengah"].shift(1).rolling(5).std()

#realized change hari ini vs kemarin, dipakai untuk naive baseline
df_model["kurs_diff1"] = df_model["Kurs Tengah"] - df_model["kurs_lag1"]

TS_FEATURES = [
    "kurs_lag1", "kurs_lag2", "kurs_lag3", "kurs_lag4", "kurs_lag5",
    "kurs_roll_mean5", "kurs_roll_std5",
]

#buang baris awal deret waktu yang belum punya histori lag/rolling lengkap
df_model = df_model.dropna(subset=TS_FEATURES + ["Target"]).reset_index(drop=True)
print(f"Baris setelah dropna fitur lag/rolling : {len(df_model)}")
df_model[["Tanggal_dt", "Kurs Tengah"] + TS_FEATURES + ["Target"]].head(5)

Baris setelah dropna fitur lag/rolling : 1196


,Tanggal_dt,Kurs Tengah,kurs_lag1,kurs_lag2,kurs_lag3,kurs_lag4,kurs_lag5,kurs_roll_mean5,kurs_roll_std5,Target
0,2021-09-08,14195.005,14239.005,14261.005,14281.005,14284.000,14306.000,14274.203,25.329746,1
1,2021-09-09,14266.000,14195.005,14239.005,14261.005,14281.005,14284.000,14252.004,36.618575,1
2,2021-09-10,14272.000,14266.000,14195.005,14239.005,14261.005,14281.005,14248.404,33.432858,0
3,2021-09-13,14225.005,14272.000,14266.000,14195.005,14239.005,14261.005,14246.603,31.419548,1
4,2021-09-14,14260.000,14225.005,14272.000,14266.000,14195.005,14239.005,14239.403,31.418975,0


# 5. Split Data Kronologis (Train/Val/Test)

In [15]:
#split 70/15/15 berurutan waktu, bukan random shuffle
n_rows = len(df_model)
train_end = int(n_rows * 0.70)
val_end = int(n_rows * 0.85)

train_df = df_model.iloc[:train_end]
val_df = df_model.iloc[train_end:val_end]
test_df = df_model.iloc[val_end:]

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Periode train : {train_df['Tanggal_dt'].min().date()} s.d. {train_df['Tanggal_dt'].max().date()}")
print(f"Periode test  : {test_df['Tanggal_dt'].min().date()} s.d. {test_df['Tanggal_dt'].max().date()}")

Train: 837 | Val: 179 | Test: 180
Periode train : 2021-09-08 s.d. 2025-02-19
Periode test  : 2025-11-24 s.d. 2026-09-01


# 6. Fungsi Evaluasi

In [16]:
def evaluate_model(y_true, y_pred, model_name):
    return {
        "model": model_name,
        "directional_accuracy": accuracy_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
    }

results = []

# 7. Persiapan Fitur & Label

In [17]:
X_train, y_train = train_df[TS_FEATURES], train_df["Target"]
X_val, y_val = val_df[TS_FEATURES], val_df["Target"]
X_test, y_test = test_df[TS_FEATURES], test_df["Target"]

# 8. Model 1 — Naive / Persistence Baseline

In [18]:
#opsi A: arah = sesuai perubahan hari sebelumnya (kurs_diff1)
def naive_predict(df):
    return (df["kurs_diff1"] > 0).astype(int)

y_pred_naive_val = naive_predict(val_df)
y_pred_naive_test = naive_predict(test_df)

results.append(evaluate_model(y_val, y_pred_naive_val, "Naive (Persistence Arah) - Val"))
results.append(evaluate_model(y_test, y_pred_naive_test, "Naive (Persistence Arah) - Test"))

#opsi B: selalu prediksi kelas mayoritas di train (pembanding lower-bound)
majority_class = int(y_train.mode()[0])
y_pred_majority_val = np.full(len(y_val), majority_class)
y_pred_majority_test = np.full(len(y_test), majority_class)

results.append(evaluate_model(y_val, y_pred_majority_val, "Naive (Majority Class) - Val"))
results.append(evaluate_model(y_test, y_pred_majority_test, "Naive (Majority Class) - Test"))

pd.DataFrame(results)


,model,directional_accuracy,f1_score
0,Naive (Persistence Arah) - Val,0.513966,0.539683
1,Naive (Persistence Arah) - Test,0.550000,0.608696
2,Naive (Majority Class) - Val,0.525140,0.688645
3,Naive (Majority Class) - Test,0.577778,0.732394


# 9. Model 2 — Logistic Regression

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train)

y_pred_logreg_val = logreg.predict(X_val_scaled)
y_pred_logreg_test = logreg.predict(X_test_scaled)

results.append(evaluate_model(y_val, y_pred_logreg_val, "Logistic Regression - Val"))
results.append(evaluate_model(y_test, y_pred_logreg_test, "Logistic Regression - Test"))

pd.DataFrame(results)


,model,directional_accuracy,f1_score
0,Naive (Persistence Arah) - Val,0.513966,0.539683
1,Naive (Persistence Arah) - Test,0.550000,0.608696
2,Naive (Majority Class) - Val,0.525140,0.688645
3,Naive (Majority Class) - Test,0.577778,0.732394
4,Logistic Regression - Val,0.547486,0.649351
5,Logistic Regression - Test,0.544444,0.549451


# 10. Ringkasan Perbandingan Seluruh Model

In [20]:
results_df = pd.DataFrame(results)
results_df["dataset"] = results_df["model"].apply(lambda x: "Val" if "Val" in x else "Test")
results_df["model"] = results_df["model"].str.replace(" - Val", "").str.replace(" - Test", "")

summary = results_df.pivot(index="model", columns="dataset", values="directional_accuracy")
summary["f1_test"] = results_df[results_df["dataset"] == "Test"].set_index("model")["f1_score"]
summary.sort_values("Test", ascending=False)


dataset,Test,Val,f1_test
model,,,
Naive (Majority Class),0.577778,0.525140,0.732394
Naive (Persistence Arah),0.550000,0.513966,0.608696
Logistic Regression,0.544444,0.547486,0.549451
